In [ ]:
# ============================================================
# Remove samples when the target sequence:
#   1. is not exactly 20 nt long, or
#   2. contains "-"
#
# Corresponding lines are also removed from:
#   full_guide_seqs.txt
#   log_transformed_activity.txt
# ============================================================

INPUT_GUIDE_FILE = "full_guide_seqs.txt"
INPUT_TARGET_FILE = "target_seqs_rc.txt"
INPUT_ACTIVITY_FILE = "log_transformed_activity.txt"

OUTPUT_GUIDE_FILE = "filtered_full_guide_seqs.txt"
OUTPUT_TARGET_FILE = "filtered_target_seqs_rc.txt"
OUTPUT_ACTIVITY_FILE = "filtered_log_transformed_activity.txt"


def read_lines(filename):
    """
    Read every line while preserving row correspondence.
    Trailing newline characters are removed.
    """
    with open(filename, "r") as file:
        return [line.rstrip("\r\n") for line in file]


# ============================================================
# 1. Read the three input files
# ============================================================

guide_lines = read_lines(INPUT_GUIDE_FILE)
target_lines = read_lines(INPUT_TARGET_FILE)
activity_lines = read_lines(INPUT_ACTIVITY_FILE)


# ============================================================
# 2. Confirm that all files have the same number of lines
# ============================================================

if not (
    len(guide_lines)
    == len(target_lines)
    == len(activity_lines)
):
    raise ValueError(
        "The three input files do not contain the same number of lines.\n"
        f"Guide file:    {len(guide_lines)} lines\n"
        f"Target file:   {len(target_lines)} lines\n"
        f"Activity file: {len(activity_lines)} lines"
    )


# ============================================================
# 3. Filter synchronized rows
# ============================================================

filtered_guides = []
filtered_targets = []
filtered_activities = []

removed_line_numbers = []
removed_wrong_length = []
removed_with_dash = []


for line_number, (guide, target, activity) in enumerate(
    zip(guide_lines, target_lines, activity_lines),
    start=1
):
    target_clean = target.strip()

    has_correct_length = len(target_clean) == 20
    contains_dash = "-" in target_clean

    # Keep only targets that are exactly 20 nt and contain no "-"
    if has_correct_length and not contains_dash:
        filtered_guides.append(guide)
        filtered_targets.append(target_clean)
        filtered_activities.append(activity)

    else:
        removed_line_numbers.append(line_number)

        if not has_correct_length:
            removed_wrong_length.append(line_number)

        if contains_dash:
            removed_with_dash.append(line_number)


# ============================================================
# 4. Write the synchronized filtered files
# ============================================================

with open(OUTPUT_GUIDE_FILE, "w") as output_file:
    for guide in filtered_guides:
        output_file.write(guide + "\n")


with open(OUTPUT_TARGET_FILE, "w") as output_file:
    for target in filtered_targets:
        output_file.write(target + "\n")


with open(OUTPUT_ACTIVITY_FILE, "w") as output_file:
    for activity in filtered_activities:
        output_file.write(activity + "\n")


# ============================================================
# 5. Print summary
# ============================================================

print("Filtering completed.")
print()

print(f"Original samples:  {len(target_lines)}")
print(f"Remaining samples: {len(filtered_targets)}")
print(f"Removed samples:   {len(removed_line_numbers)}")
print()

print(
    "Rows removed because target length was not 20 nt: "
    f"{len(removed_wrong_length)}"
)

print(
    'Rows removed because target contained "-": '
    f"{len(removed_with_dash)}"
)

print()
print("Output files:")
print(f"  {OUTPUT_GUIDE_FILE}")
print(f"  {OUTPUT_TARGET_FILE}")
print(f"  {OUTPUT_ACTIVITY_FILE}")


if removed_line_numbers:
    print()
    print("Removed original line numbers:")
    print(removed_line_numbers)